In [1]:
import sys
sys.path.append('../..')
from sqlalchemy import create_engine, MetaData, select, Table
import schedule
import pandas as pd
from datetime import datetime
import time
from config import POSTGRES_UTEA
import math

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

ENGINE = create_engine(f'postgresql+psycopg://{USER_DB}:{PASS_DB}@{HOST_DB}:{PORT_DB}/{NAME_DB}')

metadata = MetaData()
reporte_tbl = Table("reporte", metadata, autoload_with=ENGINE, schema="datos_iag")
msj_whatsapp_tbl = Table("msj_whatsapp", metadata, autoload_with=ENGINE, schema="notificaciones_wsp")

In [2]:
def crear_mensaje(datos):    
    try:
        with ENGINE.begin() as conn:
            conn.execute(msj_whatsapp_tbl.insert(), datos.to_dict(orient='records'))
        print("✅ Se ha actualizado ")
    except Exception as e:
        print("❌ Error al insertar en tabla MSJ WHASTAPP", e)
    #return df

In [3]:
df = pd.read_excel(r'G:\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\ESTIMATIVAS\V02\links_planos_estimativas_mas_cod_ca.xlsx', sheet_name = 'links')

In [4]:
df

,cod_ca,nom_ca,numero_cel,unidad_01,unidad_02,link
0,86,AGUILERA TARADELLES JOSE LUIS,67759316,13,GUAYABOCHI--AGUILERA,https://drive.google.com/file/d/1_yukzvrhb6TYq...
1,86,AGUILERA TARADELLES JOSE LUIS,67759316,55,LA CONQUISTA--AGUILERA,https://drive.google.com/file/d/1kCtyM2eJCxvOA...
2,86,AGUILERA TARADELLES JOSE LUIS,67759316,271,GUAYABOCHI 3--AGUILERA JOSE LUIS,https://drive.google.com/file/d/1waaz-_jbDg1xV...
3,86,AGUILERA TARADELLES JOSE LUIS,67759316,328,NARANJAL 5 ESTRELLAS--AGUILERA JOSE,https://drive.google.com/file/d/1dubMgn9pTRWCO...
4,86,AGUILERA TARADELLES JOSE LUIS,67759316,590,HACIENDA PETETO,https://drive.google.com/file/d/1DnOpTtMqBiOjf...
...,...,...,...,...,...,...
464,42239,CORDOVA OLGUIN LEONARDO,69005644,1136,S JUAN I--VALENTINA OLGUIN DE CORDOVA,https://drive.google.com/file/d/1ZSfTfZAtZbu4t...
465,42347,MERCADO KATERINE JUSTINIANO DE,76090266,258,MADRECITA JUNTA DEL PAILON--MERCADO,https://drive.google.com/file/d/1BIoDbbjB5vZ4X...
466,42376,HURTADO ANTONIO PEREDO,70951751,2124,TORNO DE LA VIBORA I--HURTADO ANTONIO,https://drive.google.com/file/d/12HzepHq-HDTBO...
467,42376,HURTADO ANTONIO PEREDO,78440305,2124,TORNO DE LA VIBORA I--HURTADO ANTONIO,https://drive.google.com/file/d/12HzepHq-HDTBO...


In [5]:
df['fecha_registro'] = datetime.now()

df['mensaje'] = df.apply(lambda row: f"""*{row['unidad_02']}*
{row['link']}""", axis=1)


df['enviado'] = False
df['fecha_enviado'] = None
df['numero_contac'] = '591' + df['numero_cel'].astype(str) + '@s.whatsapp.net'

In [24]:
df = df.drop(['COD_COS', 'NOMBRE', 'NUM', 'OBS'], axis=1)

In [6]:
df

,cod_ca,nom_ca,numero_cel,unidad_01,unidad_02,link,fecha_registro,mensaje,enviado,fecha_enviado,numero_contac
0,86,AGUILERA TARADELLES JOSE LUIS,67759316,13,GUAYABOCHI--AGUILERA,https://drive.google.com/file/d/1_yukzvrhb6TYq...,2026-03-29 11:30:10.872413,*GUAYABOCHI--AGUILERA*\nhttps://drive.google.c...,False,None,59167759316@s.whatsapp.net
1,86,AGUILERA TARADELLES JOSE LUIS,67759316,55,LA CONQUISTA--AGUILERA,https://drive.google.com/file/d/1kCtyM2eJCxvOA...,2026-03-29 11:30:10.872413,*LA CONQUISTA--AGUILERA*\nhttps://drive.google...,False,None,59167759316@s.whatsapp.net
2,86,AGUILERA TARADELLES JOSE LUIS,67759316,271,GUAYABOCHI 3--AGUILERA JOSE LUIS,https://drive.google.com/file/d/1waaz-_jbDg1xV...,2026-03-29 11:30:10.872413,*GUAYABOCHI 3--AGUILERA JOSE LUIS*\nhttps://dr...,False,None,59167759316@s.whatsapp.net
3,86,AGUILERA TARADELLES JOSE LUIS,67759316,328,NARANJAL 5 ESTRELLAS--AGUILERA JOSE,https://drive.google.com/file/d/1dubMgn9pTRWCO...,2026-03-29 11:30:10.872413,*NARANJAL 5 ESTRELLAS--AGUILERA JOSE*\nhttps:/...,False,None,59167759316@s.whatsapp.net
4,86,AGUILERA TARADELLES JOSE LUIS,67759316,590,HACIENDA PETETO,https://drive.google.com/file/d/1DnOpTtMqBiOjf...,2026-03-29 11:30:10.872413,*HACIENDA PETETO*\nhttps://drive.google.com/fi...,False,None,59167759316@s.whatsapp.net
...,...,...,...,...,...,...,...,...,...,...,...
464,42239,CORDOVA OLGUIN LEONARDO,69005644,1136,S JUAN I--VALENTINA OLGUIN DE CORDOVA,https://drive.google.com/file/d/1ZSfTfZAtZbu4t...,2026-03-29 11:30:10.872413,*S JUAN I--VALENTINA OLGUIN DE CORDOVA*\nhttps...,False,None,59169005644@s.whatsapp.net
465,42347,MERCADO KATERINE JUSTINIANO DE,76090266,258,MADRECITA JUNTA DEL PAILON--MERCADO,https://drive.google.com/file/d/1BIoDbbjB5vZ4X...,2026-03-29 11:30:10.872413,*MADRECITA JUNTA DEL PAILON--MERCADO*\nhttps:/...,False,None,59176090266@s.whatsapp.net
466,42376,HURTADO ANTONIO PEREDO,70951751,2124,TORNO DE LA VIBORA I--HURTADO ANTONIO,https://drive.google.com/file/d/12HzepHq-HDTBO...,2026-03-29 11:30:10.872413,*TORNO DE LA VIBORA I--HURTADO ANTONIO*\nhttps...,False,None,59170951751@s.whatsapp.net
467,42376,HURTADO ANTONIO PEREDO,78440305,2124,TORNO DE LA VIBORA I--HURTADO ANTONIO,https://drive.google.com/file/d/12HzepHq-HDTBO...,2026-03-29 11:30:10.872413,*TORNO DE LA VIBORA I--HURTADO ANTONIO*\nhttps...,False,None,59178440305@s.whatsapp.net


In [12]:
crear_mensaje(df)

✅ Se ha actualizado 


In [47]:
enviar_reporte_genencia()
while True:
    schedule.run_pending()
    time.sleep(1)

✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 
✅ Se ha actualizado 


KeyboardInterrupt: 